In [1]:
import kagglehub
path = kagglehub.dataset_download("srisyra02/ev-battery-health-prediction-dataset-20k")

100%|██████████| 2.84M/2.84M [00:00<00:00, 183MB/s]

Extracting files...


In [2]:
import os
import pandas as pd

# List all files in the downloaded Kaggle dataset path
files = [os.path.join(path, f) for f in os.listdir(path)]
print("Downloaded files:", files)

# Load the primary dataset (assuming it is a CSV file)
csv_files = [f for f in files if f.endswith('.csv')]
if csv_files:
    df = pd.read_csv(csv_files[0])
    display(df.head())
    display(df.info())
else:
    print("No CSV files found in the dataset directory.")

Downloaded files: ['/root/.cache/kagglehub/datasets/srisyra02/ev-battery-health-prediction-dataset-20k/versions/1/ev battery_failure prediction Dataset.csv']


,vehicle_id,vehicle_brand,vehicle_model,vehicle_type,manufacturing_year,battery_manufacturer,battery_chemistry,battery_capacity_kwh,drive_type,odometer_km,...,sensor_fault_count,BMS_warning_count,abnormal_voltage_events,battery_stress_index,aging_score,thermal_health_score,charging_quality_score,driving_stress_score,predicted_remaining_life_cycles,battery_failure
0,EV100000,Ford,Mustang Mach-E,Truck,2022.0,Northvolt,NMC,NaN,AWD,290844.3,...,1.0,NaN,0.0,30.37,73.05,98.14,89.73,47.11,6748.0,0
1,EV100001,Tata,Tiago EV,Truck,2017.0,CATL,NCA,38.71,RWD,182239.5,...,0.0,1.0,0.0,67.94,82.50,NaN,NaN,55.38,5211.0,0
2,EV100002,Volkswagen,ID.3,Crossover,2016.0,BYD Battery,LFP,131.29,FWD,249369.3,...,0.0,1.0,0.0,74.84,90.33,60.02,86.84,74.54,4040.0,0
3,EV100003,NaN,Kona Electric,Sedan,2019.0,SK On,NMC,132.35,FWD,134830.2,...,NaN,0.0,0.0,46.94,56.69,66.43,93.93,31.69,NaN,0
4,EV100004,MG,ZS EV,SUV,2017.0,Guoxuan,LTO,25.72,AWD,158026.0,...,1.0,2.0,3.0,49.07,73.65,95.05,100.00,23.46,7290.0,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 70 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   vehicle_id                       20000 non-null  object 
 1   vehicle_brand                    19356 non-null  object 
 2   vehicle_model                    19180 non-null  object 
 3   vehicle_type                     19356 non-null  object 
 4   manufacturing_year               19018 non-null  float64
 5   battery_manufacturer             19382 non-null  object 
 6   battery_chemistry                19278 non-null  object 
 7   battery_capacity_kwh             19310 non-null  float64
 8   drive_type                       19274 non-null  object 
 9   odometer_km                      19134 non-null  float64
 10  vehicle_age_years                19232 non-null  float64
 11  fleet_or_private                 19106 non-null  object 
 12  battery_serial    

None

In [3]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
from sklearn.preprocessing import LabelEncoder
import lightgbm as lgb

# 1. Feature Selection & Preprocessing
# Drop identifiers that are unique and shouldn't be used for prediction
ignore_cols = ['vehicle_id', 'battery_serial', 'battery_failure']
feature_cols = [col for col in df.columns if col not in ignore_cols]

X = df[feature_cols].copy()
y = df['battery_failure'].copy()

# Identify categorical and numerical columns
cat_cols = X.select_dtypes(include=['object']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object']).columns.tolist()

# Simple label encoding for categorical columns
for col in cat_cols:
    le = LabelEncoder()
    # Handle NaNs in categorical columns by filling with 'Unknown'
    X[col] = X[col].fillna('Unknown').astype(str)
    X[col] = le.fit_transform(X[col])

# Note: LightGBM natively handles missing values in numerical columns,
# so we don't strictly need to impute them, which preserves natural feature patterns.

# 2. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Model Training
model = lgb.LGBMClassifier(
    n_estimators=150,
    learning_rate=0.05,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

model.fit(
    X_train,
    y_train,
    categorical_feature=cat_cols
)

# 4. Evaluation
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

print("=== Model Performance ===")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC-AUC Score: {roc_auc_score(y_test, y_pred_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Display top 10 important features
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\n=== Top 10 Feature Importances ===")
print(importances.head(10))

=== Model Performance ===
Accuracy Score: 0.9610
ROC-AUC Score: 0.9820

Classification Report:
              precision    recall  f1-score   support

           0       0.97      0.99      0.98      3723
           1       0.81      0.57      0.67       277

    accuracy                           0.96      4000
   macro avg       0.89      0.78      0.82      4000
weighted avg       0.96      0.96      0.96      4000


=== Top 10 Feature Importances ===
previous_faults            412
BMS_warning_count          331
thermal_runaway_risk       295
thermal_health_score       245
vehicle_model              242
abnormal_voltage_events    236
battery_stress_index       210
capacity_loss_percent      198
battery_health_percent     154
state_of_health            132
dtype: int32


In [4]:
import joblib

# Save model, feature column layout, and data types so the app can load them
joblib_data = {
    'model': model,
    'feature_cols': feature_cols,
    'cat_cols': cat_cols,
    'num_cols': num_cols
}
joblib.dump(joblib_data, 'battery_model.pkl')
print("Model saved successfully!")

Model saved successfully!


In [5]:
%%writefile app.py
import streamlit as st
import pandas as pd
import joblib

st.set_page_config(page_title="EV Battery Failure Predictor", layout="wide")

# Main Panel Title
st.title("⚡ EV Battery Failure Diagnostic System")
st.write("Adjust the diagnostic metrics on the left sidebar to generate a failure prediction output in real-time.")

# Load the saved model and metadata
@st.cache_resource
def load_model_data():
    return joblib.load('battery_model.pkl')

data = load_model_data()
model = data['model']
feature_cols = data['feature_cols']
cat_cols = data['cat_cols']
num_cols = data['num_cols']

# Define inputs on the Left Sidebar
st.sidebar.header("🔌 Diagnostic Input Panel")

# Group 1: Warnings and History
st.sidebar.subheader("Alerts & Historical Faults")
previous_faults = st.sidebar.number_input("Previous Faults Count", min_value=0.0, max_value=50.0, value=0.0, step=1.0)
BMS_warning_count = st.sidebar.number_input("BMS Warning Count", min_value=0.0, max_value=50.0, value=0.0, step=1.0)
abnormal_voltage_events = st.sidebar.number_input("Abnormal Voltage Events", min_value=0.0, max_value=50.0, value=0.0, step=1.0)

# Group 2: Physical Health indicators
st.sidebar.subheader("Thermal & Degradation Metrics")
thermal_runaway_risk = st.sidebar.slider("Thermal Runaway Risk (%)", min_value=0.0, max_value=100.0, value=10.0)
thermal_health_score = st.sidebar.slider("Thermal Health Score", min_value=0.0, max_value=100.0, value=90.0)
battery_stress_index = st.sidebar.slider("Battery Stress Index", min_value=0.0, max_value=100.0, value=25.0)
capacity_loss_percent = st.sidebar.slider("Capacity Loss (%)", min_value=0.0, max_value=100.0, value=5.0)
battery_health_percent = st.sidebar.slider("Battery Health (%)", min_value=0.0, max_value=100.0, value=95.0)
state_of_health = st.sidebar.slider("State of Health (SoH %)", min_value=0.0, max_value=100.0, value=92.0)

# Create raw input record mimicking DataFrame columns
input_dict = {col: (0.0 if col in num_cols else "Unknown") for col in feature_cols}

# Bind inputs to our DataFrame keys
input_dict['previous_faults'] = previous_faults
input_dict['BMS_warning_count'] = BMS_warning_count
input_dict['abnormal_voltage_events'] = abnormal_voltage_events
input_dict['thermal_runaway_risk'] = thermal_runaway_risk
input_dict['thermal_health_score'] = thermal_health_score
input_dict['battery_stress_index'] = battery_stress_index
input_dict['capacity_loss_percent'] = capacity_loss_percent
input_dict['battery_health_percent'] = battery_health_percent
input_dict['state_of_health'] = state_of_health

# Map standard dummy label codes
for col in cat_cols:
    input_dict[col] = 0

input_df = pd.DataFrame([input_dict])

# Center Panel Output Logic
col1, col2 = st.columns([1, 1])

with col1:
    st.markdown("### Current Input Summary")
    summary_df = pd.DataFrame({
        "Metric": ["Previous Faults", "BMS Warnings", "Abnormal Voltage Events", "Thermal Runaway Risk", "Capacity Loss", "State of Health"],
        "Value": [previous_faults, BMS_warning_count, abnormal_voltage_events, f"{thermal_runaway_risk}%", f"{capacity_loss_percent}%", f"{state_of_health}%"]
    })
    st.table(summary_df)

with col2:
    st.markdown("### Predictive Output")
    if st.button("Run Diagnostic Analysis", type="primary"):
        prediction_prob = model.predict_proba(input_df)[0][1]
        prediction_class = model.predict(input_df)[0]

        st.metric(label="Failure Probability", value=f"{prediction_prob*100:.2f}%")

        if prediction_prob > 0.40:
            st.error("🚨 Critical Status: High battery failure risk detected. Immediate technical inspection recommended.")
        elif prediction_prob > 0.15:
            st.warning("⚠️ Warning Status: Moderate anomalies detected. Monitor operational parameters closely.")
        else:
            st.success("✅ Normal Status: Battery parameters are well within stable operating limits.")

Writing app.py


In [7]:
!pip install --ignore-installed blinker
!pip install streamlit

In [8]:
!pip install streamlit pyngrok

In [9]:
from pyngrok import ngrok
ngrok.set_auth_token("3H2nxZtP4iC5L9tX9K97OPLut9W_4JsZrRVF5aRFQpQCCePy1")

In [10]:
from pyngrok import ngrok

public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://vanity-amperage-commodore.ngrok-free.dev" -> "http://localhost:8501"


In [14]:
# Run the Streamlit app using Python's module syntax
# This bypasses any PATH/executable environment resolution errors in Colab
!python -m streamlit run app.py &



2026-10-26 13:02:30.164 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.106.66.222:8501

2026-10-26 13:02:41.732 Serialization of dataframe to Arrow table was unsuccessful. Applying automatic fixes for column types to make the dataframe Arrow-compatible.
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/streamlit/dataframe_util.py", line 957, in convert_pandas_df_to_arrow_table
    return pa.Table.from_pandas(df, preserve_index=preserve_index)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "pyarrow/table.pxi", line 4796, in pyarrow.lib.Table.from_pandas
  File "/usr/local/lib/python3.13/dist-packages/pyarrow/pandas_compat.py", line 651, in dataframe_to_arrays
    arrays = [convert_column(c, f)
              ~~~~~~~~~~~~~~^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pyarrow/p